<a href="https://colab.research.google.com/github/AnanyaAsthana/Machine-Learning/blob/main/habermanDecisionTree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================
# IMPORTS
# ================================
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score
)

# ================================
# LOAD KEEL .dat FILE
# ================================
def load_keel_dat(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('@'):
                data.append(line.split(','))
    return pd.DataFrame(data)

# ================================
# G-MEAN
# ================================
def g_mean(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    recalls = np.diag(cm) / cm.sum(axis=1)
    return np.sqrt(np.prod(recalls))

# ================================
# LOAD HABERMAN DATA (1st FOLD)
# ================================
train_df = load_keel_dat("haberman-10-1tra.dat")
test_df  = load_keel_dat("haberman-10-1tst.dat")

X_train = train_df.iloc[:, :-1].astype(float)
y_train = train_df.iloc[:, -1]

X_test = test_df.iloc[:, :-1].astype(float)
y_test = test_df.iloc[:, -1]

# ================================
# LABEL ENCODING
# ================================
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# ================================
# DECISION TREE MODELS
# ================================
models = {
    "ID3"  : DecisionTreeClassifier(criterion="entropy", random_state=0),
    "C4.5" : DecisionTreeClassifier(criterion="entropy", random_state=0),
    "CART" : DecisionTreeClassifier(criterion="gini", random_state=0)
}

# ================================
# TRAIN & EVALUATE
# ================================
for name, model in models.items():
    print(f"\n===== {name} =====")

    model.fit(X_train, y_train_enc)
    y_pred = model.predict(X_test)

    # Confusion Matrix
    cm = confusion_matrix(y_test_enc, y_pred)
    print("Confusion Matrix:\n", cm)

    # Precision & Recall per class
    precision = precision_score(y_test_enc, y_pred, average=None)
    recall = recall_score(y_test_enc, y_pred, average=None)

    for i, cls in enumerate(le.classes_):
        print(f"{cls} -> Precision: {precision[i]:.4f}, Recall: {recall[i]:.4f}")

    # ============================
    # AUC (FIRST CLASS AS POSITIVE)
    # ============================
    positive_class = 0

    y_test_binary = (y_test_enc == positive_class).astype(int)
    y_prob = model.predict_proba(X_test)[:, positive_class]

    auc = roc_auc_score(y_test_binary, y_prob)
    print(f"AUC (Class {le.classes_[positive_class]} as positive): {auc:.4f}")

    # ============================
    # G-MEAN
    # ============================
    print(f"G-Mean: {g_mean(y_test_enc, y_pred):.4f}")



===== ID3 =====
Confusion Matrix:
 [[15  7]
 [ 6  3]]
 negative -> Precision: 0.7143, Recall: 0.6818
 positive -> Precision: 0.3000, Recall: 0.3333
AUC (Class  negative as positive): 0.5076
G-Mean: 0.4767

===== C4.5 =====
Confusion Matrix:
 [[15  7]
 [ 6  3]]
 negative -> Precision: 0.7143, Recall: 0.6818
 positive -> Precision: 0.3000, Recall: 0.3333
AUC (Class  negative as positive): 0.5076
G-Mean: 0.4767

===== CART =====
Confusion Matrix:
 [[17  5]
 [ 7  2]]
 negative -> Precision: 0.7083, Recall: 0.7727
 positive -> Precision: 0.2857, Recall: 0.2222
AUC (Class  negative as positive): 0.4798
G-Mean: 0.4144
